# CIDS · 02 — S1: SegResNet (council member B)

Same notebook for both experiments — set `EXPERIMENT` in the first code cell:
- **`'img'`** — Experiment 1: image only (3 channels: pre, post1, post1−pre)
- **`'txt'`** — Experiment 2: + clinical instruction (`prompt_S` → BioClinicalBERT → FiLM, 30% field dropout)

**Settings:** GPU T4 x2 (one GPU used) · Internet ON · Persistence: Files.  
**Inputs:** 2-channel MAMA-MIA cache **and** `cids-assets` (final `cids_prompts.csv`; needed for `'txt'`, harmless for `'img'`).  
**Split:** `configs/splits.csv` from the repo — train 1080 / val 120 (best epoch) / test 306 (evaluated once at the end).

**Run:** run the smoke test first, then *Save Version → Save & Run All*. Training stops itself at the time budget and the test still runs. If it stopped early, attach this notebook's output as an input to a new version and set `RESUME_FROM`.

In [ ]:
EXPERIMENT = 'img'          # 'img' (Experiment 1)  or  'txt' (Experiment 2)

REPO_URL            = 'https://github.com/Vemuri-BK/cids.git'
GIT_BRANCH          = 'main'
EXPECTED_GIT_COMMIT = ''      # paste the full commit id you pushed

CACHE_ROOT  = ''              # '' = auto-detect the 2-channel cache (train/ + val/)
PROMPTS_CSV = ''              # '' = auto-detect cids_prompts.csv
RESUME_FROM = ''              # path to last.pt from a previous version's output, if continuing
TIME_BUDGET_H = 10.0          # training stops before this; test evaluation (~30-45 min) runs after

CODE_DIR = '/kaggle/working/cids'
OUT      = f'/kaggle/working/s1_{EXPERIMENT}'

In [ ]:
import os, subprocess
if os.path.isdir(f'{CODE_DIR}/.git'):
    subprocess.run(['git', '-C', CODE_DIR, 'fetch', 'origin', GIT_BRANCH], check=True)
    subprocess.run(['git', '-C', CODE_DIR, 'checkout', GIT_BRANCH], check=True)
    subprocess.run(['git', '-C', CODE_DIR, 'pull', 'origin', GIT_BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '-b', GIT_BRANCH, REPO_URL, CODE_DIR], check=True)
if EXPECTED_GIT_COMMIT:
    subprocess.run(['git', '-C', CODE_DIR, 'checkout', EXPECTED_GIT_COMMIT], check=True)
head = subprocess.check_output(['git', '-C', CODE_DIR, 'rev-parse', 'HEAD'], text=True).strip()
assert not EXPECTED_GIT_COMMIT or head == EXPECTED_GIT_COMMIT, (head, EXPECTED_GIT_COMMIT)
print('code at commit', head)
%cd {CODE_DIR}

In [ ]:
!pip install -q monai
!pytest -q tests

In [ ]:
import sys, torch, pandas as pd
from pathlib import Path
assert torch.cuda.is_available(), 'No GPU! Settings -> Accelerator -> GPU T4 x2, then restart.'
p = torch.cuda.get_device_properties(0); print(f'GPU: {p.name}, {p.total_memory/1e9:.1f} GB')
sys.path.insert(0, CODE_DIR)
from cids.data.cache import find_cache_root, list_cases
CACHE_ROOT = str(find_cache_root(CACHE_ROOT)); cases = list_cases(Path(CACHE_ROOT))
print('cache:', CACHE_ROOT, '|', len(cases), 'cases'); assert len(cases) == 1506
sp = pd.read_csv('configs/splits.csv'); print(sp.split.value_counts().to_dict())
if not PROMPTS_CSV:
    found = list(Path('/kaggle/input').rglob('cids_prompts.csv'))
    PROMPTS_CSV = str(found[0]) if found else ''
if EXPERIMENT == 'txt':
    assert PROMPTS_CSV, 'attach cids-assets'
    pr = pd.read_csv(PROMPTS_CSV, keep_default_na=False)
    assert set(sp.cache_id) <= set(pr.cache_id), 'prompt file is missing patients'
    print('prompts:', PROMPTS_CSV, '| narratives filled:', int((pr.narrative_1 != '').sum()))
TEXT = 'on' if EXPERIMENT == 'txt' else 'off'
EXTRA = f'--prompts_csv {PROMPTS_CSV}' if TEXT == 'on' else ''
print('experiment:', EXPERIMENT, '| text:', TEXT, '| out:', OUT)

## Smoke test — 6 train patients, 1 epoch, 3 val + 3 test (~5 min)
Check: no errors, loss is a normal number, a val Dice and the test summary are printed. Dice will be low here (1 short epoch) — that is expected.

In [ ]:
!python scripts/s1_train_segresnet.py --cache_root {CACHE_ROOT} --text {TEXT} {EXTRA} \
    --out /kaggle/working/s1_smoke --epochs 1 --val_every 1 --limit_train 6 --limit_eval 3
!rm -rf /kaggle/working/s1_smoke

## Full run (~8-10 h) — use *Save Version → Save & Run All*

In [ ]:
resume = f'--resume_from {RESUME_FROM}' if RESUME_FROM else ''
!python scripts/s1_train_segresnet.py --cache_root {CACHE_ROOT} --text {TEXT} {EXTRA} \
    --out {OUT} --time_budget_h {TIME_BUDGET_H} {resume}

In [ ]:
import json, matplotlib.pyplot as plt
log = pd.read_csv(f'{OUT}/train_log.csv')
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(log.epoch, log.loss); ax[0].set_title('train loss'); ax[0].set_xlabel('epoch')
v = log.dropna(subset=['val_dice']); ax[1].plot(v.epoch, v.val_dice, marker='o'); ax[1].set_title('val Dice (120)')
plt.tight_layout(); plt.show()
if os.path.exists(f'{OUT}/test_summary.json'):
    print(json.dumps(json.load(open(f'{OUT}/test_summary.json')), indent=2))
    tm = pd.read_csv(f'{OUT}/test_metrics.csv'); print(tm.groupby('collection')[['dice','hd95','nsd']].mean().round(4))
!du -sh {OUT}